In [0]:
%pip install -U langchain langchain-community databricks-langchain langchain_chroma pypdf
 
dbutils.library.restartPython()

In [0]:
from databricks_langchain import ChatDatabricks, DatabricksEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [0]:

# ==========================================
# 1. DATA PREPROCESSING
# ==========================================

loader = DirectoryLoader(
    "/Volumes/dev/bronze/raw/resumes/",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20
)

chunks = text_splitter.split_documents(documents)

print(f"Loaded pages: {len(documents)}")
print(f"Created chunks: {len(chunks)}")


# ==========================================
# 2. EMBEDDINGS
# ==========================================

embeddings = DatabricksEmbeddings(
    endpoint="databricks-bge-large-en"
)


# ==========================================
# 3. VECTOR STORE
# ==========================================

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)


# ==========================================
# 4. RETRIEVER
# ==========================================

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)


# ==========================================
# 5. LLM
# ==========================================

llm = ChatDatabricks(
    model="databricks-gpt-oss-20b"
)


# ==========================================
# 6. PROMPT
# ==========================================

prompt_template = PromptTemplate(
    input_variables=["input", "context"],
    template="""You are an AI Resume Screening Assistant designed to help HR professionals evaluate candidates.

Your task is to answer questions based ONLY on the provided resume context and its metadata.

Guidelines:

1. Answer accurately using the resume content and metadata.
2. Never invent candidate information.
3. Candidate names may be available in the document metadata under:
   candidate_name
4. If multiple resumes are present, identify and list each unique candidate separately.
5. When asked for candidate names, return all candidate names available in the retrieved context.
6. When asked about a specific candidate, use the candidate_name metadata to identify the correct resume.
7. For candidate suitability questions, clearly distinguish between:
   - Matching skills and experience
   - Missing or unclear requirements
8. If information is not available, say:
   "This information is not mentioned in the resume."
9. Do not make decisions based on sensitive personal characteristics.
10. Keep responses professional and concise.
11. Use bullet points when listing multiple candidates.

Resume Context:
{context}

HR Question:
{input}

Answer:"""
)

# ==========================================
# 7. QA CHAIN
# ==========================================

qa_chain = create_stuff_documents_chain(
    llm,
    prompt_template
)


# ==========================================
# 8. RAG CHAIN
# ==========================================

rag_chain = create_retrieval_chain(
    retriever,
    qa_chain
)


# ==========================================
# 9. CHAT
# ==========================================

print("\nChat with your own Data")

question = input("What is your query? ")

if question:
    response = rag_chain.invoke({
        "input": question
    })

    print("\nAnswer:")
    answer = response["answer"]

if isinstance(answer, list):
    text_parts = [
        item["text"]
        for item in answer
        if isinstance(item, dict) and item.get("type") == "text"
    ]
    answer = "\n".join(text_parts)

print(answer)

In [0]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

# Resume folder path
RESUME_PATH = "/Volumes/dev/bronze/raw/resumes/"

# Find all PDF files
pdf_files = list(Path(RESUME_PATH).glob("*.pdf"))

print(f"Found {len(pdf_files)} resume(s)")

documents = []

for pdf_file in pdf_files:

    print(f"Loading: {pdf_file.name}")

    loader = PyPDFLoader(str(pdf_file))
    docs = loader.load()

    # Candidate identifier from filename
    candidate_name = pdf_file.stem

    # Add metadata to every page
    for doc in docs:
        doc.metadata["candidate_name"] = candidate_name
        doc.metadata["source_file"] = pdf_file.name

    documents.extend(docs)

print(f"\nTotal pages loaded: {len(documents)}")

for pdf_file in pdf_files:
    print(f" - {pdf_file.stem}")

In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

In [0]:
# ============================================================
# 4. EMBEDDINGS
# ============================================================

embeddings = DatabricksEmbeddings(
    endpoint="databricks-bge-large-en"
)

In [0]:
# ============================================================
# 5. VECTOR STORE
# ============================================================

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="resume_hr_agent"
)

print("Vector store created successfully")

In [0]:
# ============================================================
# 7. LLM
# ============================================================

llm = ChatDatabricks(
    model="databricks-gemma-3-12b"
)

In [0]:
# ============================================================
# 8. HR RESUME PROMPT
# ============================================================

prompt_template = PromptTemplate(
    input_variables=["input", "context"],
    template="""
You are an AI Resume Screening Assistant designed for HR and recruitment teams.

You are given information retrieved from multiple candidate resumes.

Your job is to answer the HR user's question using ONLY the information available in the provided context.

IMPORTANT RULES:

1. The context may contain information from MULTIPLE resumes.

2. Each resume chunk contains metadata such as:
   - candidate_name
   - source_file

3. Treat candidate_name as the candidate identifier.

4. When the user asks for candidate names:
   - Identify ALL unique candidates available in the context.
   - Do not return only the first candidate.
   - Return each candidate only once.

5. When the user asks about a specific candidate:
   - Use the candidate_name to identify that candidate's information.

6. When comparing candidates:
   - Keep information separated by candidate.
   - Never mix experience, skills, projects, or certifications between candidates.

7. Never invent information.

8. If information is not available in the retrieved context, say:
   "This information is not available in the retrieved resumes."

9. Do NOT reveal your reasoning or internal thought process.

10. Return ONLY the final answer.

11. For multiple candidates, use a numbered list or table.

12. For a yes/no question:
   - Start with Yes or No.
   - Then give a short explanation.

13. For candidate suitability:
   - Mention matching skills/experience.
   - Mention missing or unclear requirements.
   - Do not make decisions based on sensitive personal characteristics.

14. Keep the answer professional and concise.

-----------------------------------------
RESUME CONTEXT
-----------------------------------------

{context}

-----------------------------------------
HR QUESTION
-----------------------------------------

{input}

-----------------------------------------
FINAL ANSWER
-----------------------------------------
"""
)

In [0]:
# ============================================================
# 9. DOCUMENT QA CHAIN
# ============================================================

qa_chain = create_stuff_documents_chain(
    llm,
    prompt_template
)

In [0]:
# ============================================================
# 10. RETRIEVAL RAG CHAIN
# ============================================================

rag_chain = create_retrieval_chain(
    retriever,
    qa_chain
)

In [0]:
# ============================================================
# 11. CLEAN LLM RESPONSE
# ============================================================

def extract_answer(answer):

    # Normal string response
    if isinstance(answer, str):
        return answer.strip()

    # GPT-OSS structured response
    if isinstance(answer, list):

        text_parts = []

        for item in answer:

            if isinstance(item, dict):

                if item.get("type") == "text":
                    text = item.get("text", "")

                    if text:
                        text_parts.append(text)

        return "\n".join(text_parts).strip()

    return str(answer).strip()

In [0]:
# ============================================================
# 12. CHAT WITH RESUMES
# ============================================================

print("\n========================================")
print("      RESUME HR ASSISTANT")
print("========================================")
print(f"Candidates loaded: {len(pdf_files)}")

for pdf_file in pdf_files:
    print(f" - {pdf_file.stem}")

print("\nType 'exit' to stop.\n")


while True:

    question = input("HR Query: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    if not question.strip():
        continue

    response = rag_chain.invoke({
        "input": question
    })

    answer = extract_answer(response["answer"])

    print("\nAnswer:")
    print(answer)

    print("\n" + "-" * 70 + "\n")